# Phase 4: BLS Period Search & Phase Folding
This notebook demonstrates how we perform a period search on detrended light curves using **Box Least Squares (BLS)** from `astropy.timeseries`.

### Why Box Least Squares?
- Lomb-Scargle periodograms are optimized for sinusoidal signals (like stellar pulsations or rotation).
- BLS is optimized for **box-like periodic dips**, which are characteristic of transiting exoplanets and eclipsing binaries.
- It fits a step function characterized by a period, duration, depth, and epoch ($T_0$) to the data.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from astropy.timeseries import BoxLeastSquares

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Setup complete.")

## 1. Single Star BLS Search Example
We load a detrended light curve for `TIC 231663901` and run the BLS algorithm to search for periodic transits.

In [2]:
df = pd.read_csv('../data/detrended/TIC_231663901.csv')
time = df['time'].values
flux = df['detrended_flux'].values

# Run BLS
bls = BoxLeastSquares(time, flux)
periods = bls.autoperiod(0.1, minimum_period=0.5, maximum_period=15.0, frequency_factor=5.0)
durations = np.linspace(0.01, 0.25, 10)
results = bls.power(periods, durations)

# Find best fit parameters
best_idx = np.argmax(results.power)
best_period = results.period[best_idx]
best_epoch = results.transit_time[best_idx]
best_depth = results.depth[best_idx]
best_duration = results.duration[best_idx]

print(f"Recovered Period: {best_period:.5f} days")
print(f"Recovered Depth: {best_depth*100:.3f}%")
print(f"Recovered Duration: {best_duration*24:.2f} hours")

## 2. Global Period Search Results
We load the output CSV `outputs/period_search_results.csv` containing the recovered transit parameters for all 30 targets.

In [3]:
results_df = pd.read_csv('../outputs/period_search_results.csv')
results_df[['star_id', 'label', 'catalog_period', 'recovered_period', 'relative_error', 'bls_power']].head(10)

star_id,label,catalog_period,recovered_period,relative_error,bls_power
231663901,confirmed_planet,1.430370,1.429927,0.000310,3.502241e-04
149603524,confirmed_planet,4.411937,4.412548,0.000138,2.139236e-06
336732616,confirmed_planet,3.547854,3.549894,0.000575,5.764748e-06
231670397,confirmed_planet,4.087296,4.089057,0.000431,1.669127e-07
144065872,confirmed_planet,2.184667,2.185589,0.000422,3.937971e-06
38846515,confirmed_planet,2.849383,2.849480,0.000034,1.231530e-06
92352620,confirmed_planet,3.950201,3.948171,0.000514,1.064390e-06
289793076,confirmed_planet,3.044049,3.042662,0.000456,1.125854e-04
29344935,confirmed_planet,2.766760,2.768049,0.000466,4.693972e-05
281459670,confirmed_planet,3.174350,3.176881,0.000797,1.018955e-05
